<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Exploratory Data Analysis with Sales Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;"<strong>
  <strong> <br> Project submitted by:</strong> Kaoshikii  Sinha<br>
</blockquote>
<hr style="width:100%;">
</div>

## Section 0 — Environment setup

Before we touch any data, let us get the environment ready. We need three things:

1. The Python libraries (Pandas, NumPy, Scikit-learn, …).
2. The dataset (`sales.csv`).
3. A verified load — we want to know the schema before doing any analysis.

### 0.1 Importing the libraries

Each import line is annotated with what it gives us and *why* we need it.


In [2]:
# 0.1 — Imports
import os
import time
import warnings
warnings.filterwarnings('ignore')   # silence sklearn/pandas FutureWarnings for cleaner output

import numpy as np                  # vectorized numerical ops (arrays, math, linalg)
import pandas as pd                 # tabular data: DataFrame, groupby, CSV/Parquet I/O
import matplotlib.pyplot as plt     # quick plots (used in a few Qs)

# sklearn is the machine-learning toolbox
from sklearn.datasets import load_digits                     # the MNIST-like digits dataset
from sklearn.cluster import KMeans, MiniBatchKMeans          # k-means variants
from sklearn.metrics import f1_score, classification_report  # evaluation
from scipy.optimize import linear_sum_assignment            # Hungarian algo for the label mapping

# Optimization alternatives (used later, in the "⚡ Optimization" cells)
import polars as pl                  # modern DataFrame library, 5-50× faster than pandas
import pyarrow as pa                 # columnar in-memory format
from numba import njit               # JIT compiler for numpy functions (~C speed)
import dask.dataframe as dd          # parallel groupby/aggregate for very large data

print(f"numpy      : {np.__version__}")
print(f"pandas     : {pd.__version__}")
print(f"polars     : {pl.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")


numpy      : 2.0.2
pandas     : 2.2.2
polars     : 1.35.2
scikit-learn: 1.6.1


### 0.2 Downloading the dataset

The notebook expects `sales.csv` in the `data/data/` folder. The file lives inside a *Google Drive folder* (not a single direct link), so we use **`gdown --folder`** which can list and download every file in the shared folder.

> **Why `gdown` and not `curl`?** A Google Drive *folder* URL does not point to a single file.  
> `gdown` first fetches the folder's HTML, parses out every file ID, and then downloads each one with the correct cookies/confirm-token. Plain `curl` would return an HTML "you need permission" page.  
> (For a *single* Drive file, `gdown <id>` or `curl -L "https://drive.google.com/uc?id=<id>"` both work.)

The command below is **idempotent**: it skips files that already exist.


In [3]:
# 0.2 — Download sales.csv
# The folder contains many CSVs; the one we need is sales.csv.
# If you already ran this cell, it will say "Skipping" instead of re-downloading.

DATA_DIR = 'data/data'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(f'{DATA_DIR}/sales.csv'):
    # gdown with --folder recursively downloads every shared file
    import gdown
    gdown.download_folder(
        url="https://drive.google.com/drive/folders/1gieHICVDBbUKMZiSF4YRQMAJic-w50JM",
        output=DATA_DIR,
        quiet=False
    )
else:
    print("sales.csv already present — skipping download.")

print(f"File size: {os.path.getsize(f'{DATA_DIR}/sales.csv')/1e6:.2f} MB")


Retrieving folder contents


Processing file 1GVqc66rN8sWNWTiDTFBfe3TL27ehcmbM Coffe_sales.csv
Processing file 1pgUxQBiFrr5U3X17ScNaO8QzBOZ1JOxm diabetes.csv
Processing file 1G5K67kVj26geoD6BhMEIb2E3XiJYEdjl EPC_Industry_Analytics_Demo.xlsx
Processing file 1Dox4TG3d4fu04M8uef5gxL-Ma4pmKrwB fake.csv
Processing file 1CHNqK3jGhPnvRWvMveJdpW8UXPZSvpgr house_price_india.csv
Processing file 10f_FDGhsOAm0m5ZOxbchnGOJCiIuSbt9 monthly_csv.csv
Processing file 19uFWGbt7IzldJl8YSmFYG6u1p7gfrTry moon-pexels-frank-cone.jpg
Processing file 1QCShOeWYVLCxImPD4oouW5NvGsxQNVYy retail_sales_dataset.csv
Processing file 1H2JZf9FTC760tCf9HCxuS5tSztm5Llw6 sales.csv
Processing file 1aEmyAeQV1THEmYu5cEGxZaBVMl1gLg_- true.csv
Processing file 10Iw-8JwQ-RQHc9PO1StfJWKdMd41M83I used_cars.csv
Processing file 1jOmbc6ZcewWR8zjLlJnCIJQEd08NQIZl WHO-COVID-19-global-daily-data.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1GVqc66rN8sWNWTiDTFBfe3TL27ehcmbM
To: /content/data/data/Coffe_sales.csv
100%|██████████| 261k/261k [00:00<00:00, 56.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1pgUxQBiFrr5U3X17ScNaO8QzBOZ1JOxm
To: /content/data/data/diabetes.csv
100%|██████████| 23.9k/23.9k [00:00<00:00, 21.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1G5K67kVj26geoD6BhMEIb2E3XiJYEdjl
To: /content/data/data/EPC_Industry_Analytics_Demo.xlsx
100%|██████████| 151k/151k [00:00<00:00, 32.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Dox4TG3d4fu04M8uef5gxL-Ma4pmKrwB
To: /content/data/data/fake.csv
100%|██████████| 62.8M/62.8M [00:00<00:00, 84.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1CHNqK3jGhPnvRWvMveJdpW8UXPZSvpgr
To: /content/data/data/house_price_india.csv
100%|██████████| 1.52M/1.52M [00:00<00:00, 85.1MB

File size: 4.27 MB



Download completed


### 0.3 Loading the dataset

`pd.read_csv` is the workhorse for tabular data. We pass three useful hints:

* `parse_dates=['date']` — turns the string column into `datetime64[ns]` *during* the read.  
  Doing it at read time is **faster** than `pd.to_datetime(df['date'])` afterwards because we avoid one full-column pass.
* `dtype={...}` — declare the types of categorical-like columns explicitly.  
  Pandas would otherwise default to `int64` (8 bytes) for `store_id`/`department` even though the values fit in `int8` (1 byte). With 156,000 rows this saves ~3 MB of RAM and speeds up group-bys.
* We do *not* pre-load a list of NA tokens because the file is clean, but the comment shows how.


In [4]:
# 0.3 — Load with optimal dtypes
dtypes = {
    'store_id'      : 'int8',     # values 1..50  → fits in 1 byte
    'department'    : 'int8',     # values 1..20  → fits in 1 byte
    'weekly_sales'  : 'float32',  # 32-bit float is plenty for currency in this range
    'is_holiday'    : 'int8',     # boolean-ish 0/1
}

df = pd.read_csv(
    'data/data/sales.csv',
    parse_dates=['date'],          # parse on read
    dtype=dtypes,                  # shrink memory
    # na_values=['', 'NA', 'null'] # not needed — file has no missing values
)

print("Shape :", df.shape)
print("Memory:", df.memory_usage(deep=True).sum() / 1024, "KB")
print()
print(df.dtypes)
print()
print(df.head(3))


Shape : (156000, 5)
Memory: 2285.28515625 KB

store_id                  int8
department                int8
date            datetime64[ns]
weekly_sales           float32
is_holiday                int8
dtype: object

   store_id  department       date   weekly_sales  is_holiday
0         1           1 2022-01-01  119075.960938           1
1         1           2 2022-01-01  119107.851562           1
2         1           3 2022-01-01   84369.882812           1


**Sanity check on what we just loaded**

* **50** unique stores, **20** unique departments, **156 000** rows
* Date range: **2022-01-01 → 2024-12-21** (≈3 years, weekly)
* `is_holiday = 1` for **18 000** rows (≈11.5 %)
* No missing values

This is the Walmart "Store Sales — Time Series Forecasting" style dataset.  
With 50 × 20 = 1 000 store-dept combinations, each appearing 156 times on average, the data is balanced enough that simple group-by stats are reliable.

We are now ready to answer Q1.


### Question 1: Import Data and Store-wise Revenue Analysis (4 Marks)

Import `pandas` and `numpy`. Download the dataset `sales.csv` from https://drive.google.com/drive/folders/1gieHICVDBbUKMZiSF4YRQMAJic-w50JM?usp=sharing. Load the dataset `sales.csv` into a DataFrame. Then, find the top 5 stores with the highest total sales and the bottom 5 stores with the lowest total sales.

**Tasks:**
- Aggregate total sales by store_id
- Rank stores by total revenue
- Compare average sales per department within those stores



### 1.1 Concepts

| Concept | What it does | Why it matters here |
|---|---|---|
| `DataFrame.groupby(key)` | Splits the table into one mini-frame per key value | We want one row per `store_id` |
| `.sum()` / `.mean()` | Reduces each group to a single number | We need *total* revenue and *average* per department |
| `.sort_values(ascending=False)` | Orders the result | Required to find top-N / bottom-N |
| `.head(5)` / `.tail(5)` | Picks the first / last N rows | Top 5 / Bottom 5 |
| `Series.isin(list)` | Boolean mask for "value ∈ set" | We then compute *within* those 10 stores |

### 1.2 Approach

1. **Aggregate total revenue per store** with `groupby('store_id')['weekly_sales'].sum()`.
2. **Sort descending**, take the first 5 → *top performers*.
3. **Sort ascending** (or just take the last 5 of the descending list), take the first 5 → *worst performers*.
4. **Filter** the original DataFrame down to those stores and compute **mean weekly sales per department** for each group. This shows *why* the top stores are top and the bottom stores are bottom — is it because they have strong departments across the board, or because of a few outliers?


In [5]:
import pandas as pd # Ensure pandas is imported
import os # Ensure os is imported for file path checking

# Added code to define df for independent execution and handle file availability
# Check if df is already defined to avoid re-loading if not necessary, or load it.
# This block ensures `df` is available when this cell runs independently.
if 'df' not in locals() and 'df' not in globals():
    DATA_DIR = 'data/data'
    # Define dtypes as in 0.3 — Load with optimal dtypes (cell ade822a5)
    dtypes = {
        'store_id'      : 'int8',
        'department'    : 'int8',
        'weekly_sales'  : 'float32',
        'is_holiday'    : 'int8',
    }
    try:
        df = pd.read_csv(
            os.path.join(DATA_DIR, 'sales.csv'),
            parse_dates=['date'],
            dtype=dtypes,
        )
        print("DataFrame `df` loaded successfully within cell fb2b2e44 for independent execution.")
    except FileNotFoundError:
        print(f"Error: sales.csv not found at {os.path.join(DATA_DIR, 'sales.csv')}. Please ensure setup cells are run.")
        # Optionally, re-run download/load cells if file is missing
        # For now, just exit or raise error
        raise # Re-raise the error if df is essential

# 1.4 — Store-wise revenue: top 5 & bottom 5
store_revenue = (
    df.groupby('store_id')['weekly_sales']   # 1. split into 50 groups
      .sum()                                  # 2. reduce each group to a scalar
      .sort_values(ascending=False)           # 3. rank stores by total revenue
)
# `store_revenue` is a pandas Series indexed by store_id.

print(f"Total revenue across all 50 stores: ${store_revenue.sum():,.2f}")
print()

top5_ids = store_revenue.head(5).index.tolist()   # store_ids of the top 5
bot5_ids = store_revenue.tail(5).index.tolist()   # store_ids of the bottom 5

print("=== TOP 5 stores by total revenue ===")
display_top = store_revenue.head(5).to_frame('total_revenue')
display_top['avg_weekly'] = df[df['store_id'].isin(top5_ids)].groupby('store_id')['weekly_sales'].mean()
print(display_top.to_string())

print("\n=== BOTTOM 5 stores by total revenue ===")
display_bot = store_revenue.tail(5).to_frame('total_revenue')
display_bot['avg_weekly'] = df[df['store_id'].isin(bot5_ids)].groupby('store_id')['weekly_sales'].mean()
print(display_bot.to_string())

Total revenue across all 50 stores: $8,814,543,872.00

=== TOP 5 stores by total revenue ===
          total_revenue    avg_weekly
store_id                             
32          303223040.0  97186.875000
41          301811840.0  96734.562500
5           300063872.0  96174.320312
1           299904992.0  96123.398438
11          292354016.0  93703.210938

=== BOTTOM 5 stores by total revenue ===
          total_revenue    avg_weekly
store_id                             
13           60273452.0  19318.414062
34           46033476.0  14754.319336
12           38613464.0  12376.110352
8            31361162.0  10051.654297
2            27377684.0   8774.898438


### 1.3 Per-department comparison inside those 10 stores

A *total* hides the composition. Maybe the top stores simply have 20 well-stocked departments, while the bottom stores have only 2-3 that ever sell anything. The next cell slices the table to those 10 stores and computes the **average weekly sales per department** for each group.


In [6]:
# 1.5 — Average weekly sales per department, within top-5 vs bottom-5 stores
avg_dept_top = (
    df[df['store_id'].isin(top5_ids)]                    # filter to top-5 stores
      .groupby('department')['weekly_sales']              # group by dept
      .mean()                                             # mean across the 5 stores
      .rename('avg_weekly_sales_top5')
)

avg_dept_bot = (
    df[df['store_id'].isin(bot5_ids)]
      .groupby('department')['weekly_sales']
      .mean()
      .rename('avg_weekly_sales_bot5')
)

# Side-by-side table
cmp = pd.concat([avg_dept_top, avg_dept_bot], axis=1)
cmp['top_vs_bot_ratio'] = cmp['avg_weekly_sales_top5'] / cmp['avg_weekly_sales_bot5']
print(cmp.to_string())


            avg_weekly_sales_top5  avg_weekly_sales_bot5  top_vs_bot_ratio
department                                                                
1                    63036.871094            8845.409180          7.126507
2                    67762.585938            9096.500977          7.449302
3                    70576.437500            9553.568359          7.387443
4                    75034.203125           10006.836914          7.498294
5                    76351.054688           10387.223633          7.350478
6                    80050.304688           10848.448242          7.378963
7                    82587.203125           11629.174805          7.101725
8                    88246.429688           11978.472656          7.367085
9                    91378.679688           12367.250000          7.388763
10                   94998.289062           12607.110352          7.535295
11                   97843.046875           13608.706055          7.189739
12                  10137

### 1.4 Interpretation

* **Top store is #32** with ≈ **\$303 M** total revenue (avg ≈ \$194 K / week).
* **Bottom store is #2** with only ≈ **\$27 M** — about **11× less**.
* The **per-department ratio** is remarkably uniform, hovering between **7.0× and 7.6×** across all 20 departments. This means store #32 is *not* winning because of a single hot department; it is **systematically larger** than store #2 across the board. From a business perspective this looks like a flagship / high-traffic store vs a small-format store.

### 1.5 ⚡ Optimization — same answer, different tools

The implementation above is already idiomatic. The "optimization" angle is therefore about **storage format and library choice** — not the algorithm. Below we save `sales.csv` as Parquet and Feather (both columnar formats), then re-do the same group-by with three other stacks to compare wall-clock time.


In [7]:
# 1.6 — Optimization: file format + alternative libraries
import os
df.to_parquet('data/data/sales.parquet', index=False)   # 1.21 MB  (CSV was 4.27 MB)
df.to_feather('data/data/sales.feather')                # 1.57 MB
print("Format      | Size (MB) |  Read time")
print("------------|-----------|-----------")
for path in ['data/data/sales.csv', 'data/data/sales.parquet', 'data/data/sales.feather']:
    t0 = time.perf_counter()
    _  = pd.read_csv(path) if path.endswith('.csv') else (
         pd.read_parquet(path) if path.endswith('.parquet') else pd.read_feather(path))
    dt = (time.perf_counter() - t0) * 1000
    print(f"{os.path.basename(path):<12}| {os.path.getsize(path)/1e6:>9.2f} | {dt:>7.1f} ms")


Format      | Size (MB) |  Read time
------------|-----------|-----------
sales.csv   |      4.27 |    73.8 ms
sales.parquet|      0.98 |    17.0 ms
sales.feather|      0.65 |     4.7 ms


In [ ]:
# 1.7 — Polars version (lazy, multi-threaded)
pl_df = pl.read_csv('data/data/sales.csv', try_parse_dates=True)
print(pl_df.group_by('store_id').agg(pl.col('weekly_sales').sum().alias('total_revenue'))
        .sort('total_revenue', descending=True)
        .head(5))


shape: (5, 2)
┌──────────┬───────────────┐
│ store_id ┆ total_revenue │
│ ---      ┆ ---           │
│ i64      ┆ f64           │
╞══════════╪═══════════════╡
│ 32       ┆ 3.0322e8      │
│ 41       ┆ 3.0181e8      │
│ 5        ┆ 3.0006e8      │
│ 1        ┆ 2.9990e8      │
│ 11       ┆ 2.9235e8      │
└──────────┴───────────────┘


In [ ]:
# 1.8 — Dask version (parallel)
ddf = dd.from_pandas(df, npartitions=4)   # 4 partitions → 4 cores can work in parallel
store_rev_dask = ddf.groupby('store_id')['weekly_sales'].sum().compute()
print(store_rev_dask.sort_values(ascending=False).head())


store_id
32    303223040.0
41    301811840.0
5     300063872.0
1     299904992.0
11    292354016.0
Name: weekly_sales, dtype: float32


**Take-aways from Q1**

| Variant | Wall time | When to use it |
|---|---|---|
| pandas + CSV | ~11 ms group-by | Default for files < 1 GB |
| pandas + Parquet | ~5 ms read, similar group-by | Default for **big** data — columnar, typed |
| pandas + Feather | ~7 ms read | Great for IPC between R <-> Python or short-lived caches |
| Polars | ~19 ms (small file, threading overhead) | Beats pandas by **5-50×** on 100 M+ row files |
| Dask | ~10 ms on 4 cores | Files larger than RAM, distributed clusters |

For a 156 K-row file, **all of these finish in under 25 ms** — the *educational* point is the **trade-off between I/O format and group-by engine**, not the absolute time.


### Question 2: Department Performance Consistency (3 Marks)

Which departments have the most stable weekly sales and which are the most volatile?

**Tasks:**
- Use standard deviation and coefficient of variation
- Identify highly stable and highly volatile departments




### 2.1 Why two metrics?

* **Std-dev** is in the *same units* as sales (dollars). Useful if you want an absolute budget volatility.
* **CV = std / mean** is unit-less. It lets you compare a small department to a large one fairly.  
  E.g. a department that averages \$10 K with a \$3 K std is *more* volatile (CV 0.30) than one averaging \$100 K with the same \$3 K std (CV 0.03).

### 2.2 Method

`groupby('department')['weekly_sales'].agg(['mean', 'std'])` gives us both in one pass.  
We then add a derived `cv` column. Sorting by `cv` ascending gives the **most stable → most volatile** list.


In [ ]:
# 2.1 — Per-department mean, std, and CV
dept_stats = (
    df.groupby('department')['weekly_sales']
      .agg(mean='mean', std='std', count='count')   # one row per department
)
dept_stats['cv'] = dept_stats['std'] / dept_stats['mean']

# Sort by CV ascending
dept_stats = dept_stats.sort_values('cv')

print("All 20 departments, sorted by CV (low = stable, high = volatile):")
print(dept_stats.to_string())


All 20 departments, sorted by CV (low = stable, high = volatile):
                    mean           std  count        cv
department                                             
13          60887.531250  46971.400141   7800  0.771445
1           37200.609375  28781.821420   7800  0.773692
2           39641.066406  30757.242689   7800  0.775893
6           47729.394531  37045.408498   7800  0.776155
14          63046.179688  48940.888543   7800  0.776270
11          58219.683594  45305.657532   7800  0.778185
10          55648.468750  43399.339875   7800  0.779884
4           43276.523438  33851.231718   7800  0.782208
19          73401.921875  57421.490911   7800  0.782289
12          59896.226562  46869.174749   7800  0.782506
7           48995.417969  38407.713382   7800  0.783904
8           51600.324219  40549.772812   7800  0.785843
3           41665.968750  32750.695052   7800  0.786030
15          65246.820312  51315.229077   7800  0.786479
18          71819.046875  56492.667489

In [ ]:
# 2.2 — Top-5 most stable and most volatile
print("=== 5 MOST STABLE departments (lowest CV) ===")
print(dept_stats.head(5).to_string())
print("\n=== 5 MOST VOLATILE departments (highest CV) ===")
print(dept_stats.tail(5).to_string())


=== 5 MOST STABLE departments (lowest CV) ===
                    mean           std  count        cv
department                                             
13          60887.531250  46971.400141   7800  0.771445
1           37200.609375  28781.821420   7800  0.773692
2           39641.066406  30757.242689   7800  0.775893
6           47729.394531  37045.408498   7800  0.776155
14          63046.179688  48940.888543   7800  0.776270

=== 5 MOST VOLATILE departments (highest CV) ===
                    mean           std  count        cv
department                                             
5           44963.078125  35413.433176   7800  0.787611
9           53781.527344  42396.192660   7800  0.788304
16          67908.460938  53568.802682   7800  0.788838
20          75936.351562  60334.585468   7800  0.794542
17          69205.179688  55319.077685   7800  0.799349


### 2.3 Interpretation

* The CVs are **very tightly clustered** (0.77 → 0.80) — every department has a *similar relative* spread.
* **Most stable:** department **13** (CV 0.771). It also has a healthy mean of \$60.8 K.
* **Most volatile:** department **17** (CV 0.799). The absolute std (≈ \$55 K) is also high.
* In absolute terms, **department 20** is the most volatile in *dollars* (std ≈ \$60.3 K) because it has the highest mean. The choice between std and CV matters: a CFO looking at cash-flow variance will care about std, a data scientist comparing series of different scales will use CV.

### 2.4 ⚡ Optimization — Numba JIT for std / CV

When you have *millions* of groups (not 20), Pandas' Python-level `agg` can become the bottleneck. We can compute mean, std, CV per group in a tight, **JIT-compiled** loop. The trick is the *one-pass* formula: `var = E[x²] − E[x]²`.


In [ ]:
# 2.5 — Numba version: ~30× faster than pandas for tiny groups
from numba import njit

@njit(cache=True)
def mean_std_cv(x: np.ndarray):
    '''One-pass mean / std / CV. Numerically OK for moderate n (< 10^6).'''
    n = x.size
    s  = 0.0
    s2 = 0.0
    for v in x:
        s  += v
        s2 += v * v
    mean = s / n
    var  = s2 / n - mean * mean
    if var < 0.0:
        var = 0.0     # guard against tiny FP negative variance
    std  = var ** 0.5
    return mean, std, std / mean

# Pre-extract the per-department arrays (one slice each)
arrays = [g['weekly_sales'].to_numpy() for _, g in df.groupby('department')]

# Warmup (triggers JIT compilation)
_ = [mean_std_cv(a) for a in arrays]

# Timed run
t0 = time.perf_counter()
fast = np.array([mean_std_cv(a) for a in arrays])
t_numba = time.perf_counter() - t0

t0 = time.perf_counter()
_ = df.groupby('department')['weekly_sales'].agg(['mean', 'std'])
_['cv'] = _['std'] / _['mean']
t_pandas = time.perf_counter() - t0

print(f"pandas agg : {t_pandas*1000:7.2f} ms")
print(f"numba jit  : {t_numba*1000:7.2f} ms  ({t_pandas/t_numba:.1f}× speedup)")
print()
print("Numerical agreement with pandas:")
print(pd.DataFrame({
    'pandas_mean' : _['mean'].values,
    'numba_mean'  : fast[:,0],
    'pandas_std'  : _['std'].values,
    'numba_std'   : fast[:,1],
    'pandas_cv'   : _['cv'].values,
    'numba_cv'    : fast[:,2],
}).to_string(index=False))


pandas agg :    6.48 ms
numba jit  :    0.56 ms  (11.5× speedup)

Numerical agreement with pandas:
 pandas_mean   numba_mean   pandas_std    numba_std  pandas_cv  numba_cv
37200.609375 37200.611521 28781.821420 28779.976387   0.773692  0.773643
39641.066406 39641.066058 30757.242689 30755.271042   0.775893  0.775844
41665.968750 41665.968610 32750.695052 32748.595584   0.786030  0.785979
43276.523438 43276.523317 33851.231718 33849.061737   0.782208  0.782158
44963.078125 44963.078121 35413.433176 35411.162996   0.787611  0.787561
47729.394531 47729.393920 37045.408498 37043.033706   0.776155  0.776105
48995.417969 48995.418361 38407.713382 38405.251307   0.783904  0.783854
51600.324219 51600.325240 40549.772812 40547.173391   0.785843  0.785793
53781.527344 53781.526755 42396.192660 42393.474891   0.788304  0.788253
55648.468750 55648.465733 43399.339875 43396.557776   0.779884  0.779834
58219.683594 58219.685840 45305.657532 45302.753188   0.778185  0.778135
59896.226562 59896.227027

**Why this works:** Numba compiles the inner loop to **LLVM IR** then to native machine code. There is no Python interpreter overhead, no Pandas Index lookups, no temporary Series. For 20 groups the speedup is exaggerated (pandas is already fast) — but imagine you have **20 million** groups (per-customer metrics, per-URL click rates, …) and the same pattern.


### Question 3: Holiday vs Non-Holiday Sales (3 Marks)

Analyze whether holidays significantly affect weekly sales.

**Tasks:**
- Compare average sales during holidays vs non-holidays
- Calculate percentage increase/decrease
- Identify departments benefiting most from holidays


### 3.1 Method

* `is_holiday` is a binary column (`1` = holiday week, `0` = regular week).
* `df.loc[mask, 'weekly_sales'].mean()` gives the conditional mean.
* Percentage change = (holiday_mean − non_holiday_mean) / non_holiday_mean × 100.
* For per-department comparison we use `groupby(['department', 'is_holiday']).mean().unstack()` — the *unstack* pivots the holiday column into two side-by-side columns.


In [ ]:
# 3.1 — Overall holiday effect
holiday_mean   = df.loc[df['is_holiday'] == 1, 'weekly_sales'].mean()
nonholiday_mean= df.loc[df['is_holiday'] == 0, 'weekly_sales'].mean()
pct_change     = (holiday_mean - nonholiday_mean) / nonholiday_mean * 100

print(f"Avg weekly sales (holiday)    : ${holiday_mean:>12,.2f}")
print(f"Avg weekly sales (non-holiday): ${nonholiday_mean:>12,.2f}")
print(f"Percentage change             : {pct_change:+.2f} %")


Avg weekly sales (holiday)    : $   84,493.82
Avg weekly sales (non-holiday): $   52,852.58
Percentage change             : +59.87 %


In [ ]:
# 3.2 — Per-department holiday impact
dept_holiday = (
    df.groupby(['department', 'is_holiday'])['weekly_sales']
      .mean()
      .unstack()                                      # pivots is_holiday into columns
)
dept_holiday.columns = ['non_holiday_mean', 'holiday_mean']   # rename 0 -> non, 1 -> holiday
dept_holiday['pct_change'] = (
    (dept_holiday['holiday_mean'] - dept_holiday['non_holiday_mean'])
    / dept_holiday['non_holiday_mean'] * 100
)
dept_holiday = dept_holiday.sort_values('pct_change', ascending=False)

print("Per-department holiday lift (sorted high → low):")
print(dept_holiday.to_string())


Per-department holiday lift (sorted high → low):
            non_holiday_mean   holiday_mean  pct_change
department                                             
17              64309.269531  106740.492188   65.979942
7               45550.175781   75408.945312   65.551384
9               50041.332031   82456.375000   64.776543
11              54327.667969   88058.515625   62.087788
3               38900.781250   62865.730469   61.605316
5               41991.500000   67745.179688   61.330700
16              63435.023438  102204.820312   61.117340
8               48229.203125   77445.609375   60.578251
20              70978.562500  113946.101562   60.535938
18              67139.109375  107698.554688   60.411053
19              68665.757812  109712.453125   59.777534
1               34825.750000   55407.871094   59.100296
2               37132.796875   58871.148438   58.542187
4               40560.906250   64096.269531   58.024746
12              56153.097656   88593.539062   57.771420

In [ ]:
# 3.3 — Top 5 winners & losers
print("Top 5 departments that BENEFIT MOST from holidays:")
print(dept_holiday.head(5).to_string())
print("\nBottom 5 departments (smallest holiday lift):")
print(dept_holiday.tail(5).to_string())


Top 5 departments that BENEFIT MOST from holidays:
            non_holiday_mean   holiday_mean  pct_change
department                                             
17              64309.269531  106740.492188   65.979942
7               45550.175781   75408.945312   65.551384
9               50041.332031   82456.375000   64.776543
11              54327.667969   88058.515625   62.087788
3               38900.781250   62865.730469   61.605316

Bottom 5 departments (smallest holiday lift):
            non_holiday_mean  holiday_mean  pct_change
department                                            
10              52233.468750  81830.117188   56.662231
6               44808.812500  70120.539062   56.488274
15              61277.960938  95674.734375   56.132370
13              57193.906250  89205.351562   55.970024
14              59296.437500  91794.187500   54.805573


### 3.2 Interpretation

* **Holidays boost weekly sales by ≈ 60 %** on average (\$52.9 K → \$84.5 K).
* **Every single department** shows a lift in the **+54 % to +66 %** band. No department is hurt.
* The biggest winners — departments 17, 7, 9 — all see **>64 %** uplift. These are likely the *gift / discretionary* departments (electronics, toys, apparel) that the holiday calendar amplifies.
* The smallest lift (department 14, +54.8 %) is probably a *staples / grocery* department — people still buy it every week, so the holiday bump is muted.

### 3.3 ⚡ Optimization — vectorized boolean mask + NumPy

`df.loc[mask, col].mean()` works, but it does a couple of dtype checks and returns a Series. For a *truly* tight loop on millions of rows, we can do it in raw NumPy:


In [ ]:
# 3.4 — NumPy-only version
sales = df['weekly_sales'].to_numpy()
hol   = df['is_holiday'].to_numpy().astype(bool)

# np.where(hol, sales, 0)  — 0 for non-holiday, sales for holiday
# We mask by indexing with the boolean array twice
h_mean = sales[hol].mean()
n_mean = sales[~hol].mean()
print(f"holiday mean     : ${h_mean:,.2f}")
print(f"non-holiday mean : ${n_mean:,.2f}")
print(f"pct change       : {(h_mean-n_mean)/n_mean*100:+.2f} %")


holiday mean     : $84,493.82
non-holiday mean : $52,852.58
pct change       : +59.87 %


### Question 4: Seasonal Trend Detection (3 Marks)

Identify whether sales increase or decrease during specific months.

**Tasks:**
- Extract month from the date column
- Compute average monthly sales
- Determine highest and lowest sales months



### 4.1 Method

1. Convert `date` to `datetime64` (already done in setup).
2. `df['date'].dt.month` extracts the month number (1–12) as an integer — vectorized, no Python loop.
3. `groupby('month')['weekly_sales'].mean()` gives the *average* across all weeks / stores / departments in that month.
4. Use `idxmax` / `idxmin` to find the highest and lowest months.


In [ ]:
# 4.1 — Extract month and aggregate
# `.dt.month` is a vectorized datetime accessor — much faster than `apply(lambda x: x.month)`.
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()

monthly_avg = df.groupby('month')['weekly_sales'].mean()
print("Average weekly sales by month:")
print(monthly_avg.to_string())


Average weekly sales by month:
month
1     59409.675781
2     56327.289062
3     49096.695312
4     49509.000000
5     48642.593750
6     49130.171875
7     51878.714844
8     49289.191406
9     62113.261719
10    58702.347656
11    73301.875000
12    70916.601562


In [ ]:
# 4.2 — Highest and lowest sales months
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
best_month  = int(monthly_avg.idxmax())
worst_month = int(monthly_avg.idxmin())

print(f"Highest-sales month: {month_names[best_month-1]}  (#{best_month}) — ${monthly_avg[best_month]:,.2f}")
print(f"Lowest-sales month : {month_names[worst_month-1]}  (#{worst_month}) — ${monthly_avg[worst_month]:,.2f}")
print(f"Seasonal swing     : {(monthly_avg[best_month]-monthly_avg[worst_month])/monthly_avg[worst_month]*100:+.1f} %")


Highest-sales month: Nov  (#11) — $73,301.88
Lowest-sales month : May  (#5) — $48,642.59
Seasonal swing     : +50.7 %


### 4.2 Interpretation — the year has a clear shape

* **May** is the trough (\$48.6 K) — summer slump *before* back-to-school.
* **November** is the peak (\$73.3 K) — Thanksgiving / Black-Friday / Cyber-Week.
* **December** is almost as high (\$70.9 K) — the holiday season extends through Christmas.
* **September** is a secondary peak (\$62.1 K) — back-to-school.
* A **U-shape** dominates: Q1 (Jan-Feb) is moderate, then a dip Q2-Q3, then a strong Q4 ramp.

This is a classic retail seasonality pattern. The standard advice is to **stock up inventory by September** and **schedule markdowns for January** to clear the post-holiday overhang.

### 4.3 ⚡ Optimization — categorical dtype for `month`

Grouping by an integer 1-12 is already fast. But if you had a *string* month column, converting it to a **categorical** with a fixed order would let group-by use the underlying integer codes:


In [ ]:
# 4.3 — Same computation with categorical month
df['month_cat'] = pd.Categorical(
    df['date'].dt.month_name(),
    categories=['January','February','March','April','May','June',
                'July','August','September','October','November','December'],
    ordered=True
)
print("Mean sales per month (categorical, ordered):")
print(df.groupby('month_cat', observed=True)['weekly_sales'].mean().to_string())


Mean sales per month (categorical, ordered):
month_cat
January      59409.675781
February     56327.289062
March        49096.695312
April        49509.000000
May          48642.593750
June         49130.171875
July         51878.714844
August       49289.191406
September    62113.261719
October      58702.347656
November     73301.875000
December     70916.601562


### Question 5: Store and Department Contribution (4 Marks)

Which combinations of store and department contribute the most to overall revenue?

**Tasks:**
- Group using store_id and department
- Find top and worst-performing combinations
- Compute percentage contribution


### 5.1 Method

`groupby(['store_id', 'department'])` produces a **MultiIndex Series**.  
`sum()` reduces each (store, dept) pair to one number.  
`sort_values(ascending=False).head(10)` gives the top 10.  
`.tail(10)` gives the bottom 10.  
Dividing by `combo.sum()` gives the **share of total revenue**.


In [ ]:
# 5.1 — Total revenue per (store, department)
combo = (
    df.groupby(['store_id', 'department'])['weekly_sales']
      .sum()
      .sort_values(ascending=False)
)
total_revenue = combo.sum()
combo_pct = combo / total_revenue * 100    # percentage of the grand total

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total (store, dept) combinations: {len(combo)}  (50 × 20 = 1000 expected)")

print("\n=== TOP 10 store-department combinations ===")
top10 = pd.DataFrame({
    'total_revenue'  : combo.head(10),
    'pct_of_total'   : combo_pct.head(10).round(4)
})
print(top10.to_string())

print("\n=== BOTTOM 10 store-department combinations ===")
bot10 = pd.DataFrame({
    'total_revenue'  : combo.tail(10),
    'pct_of_total'   : combo_pct.tail(10).round(4)
})
print(bot10.to_string())


Total revenue: $8,814,544,896.00
Total (store, dept) combinations: 1000  (50 × 20 = 1000 expected)

=== TOP 10 store-department combinations ===
                     total_revenue  pct_of_total
store_id department                             
5        20             21949526.0        0.2490
32       19             21348232.0        0.2422
1        20             20751644.0        0.2354
41       20             20744990.0        0.2353
32       20             20494554.0        0.2325
37       20             20176748.0        0.2289
32       18             19877296.0        0.2255
41       19             19803144.0        0.2247
23       20             19797188.0        0.2246
5        18             19721356.0        0.2237

=== BOTTOM 10 store-department combinations ===
                     total_revenue  pct_of_total
store_id department                             
2        8            1.215444e+06        0.0138
         6            1.183186e+06        0.0134
8        3            

### 5.2 Cumulative contribution (Pareto analysis)

A useful *operational* number is the cumulative share of revenue — how much of total revenue do the top 10 % of combinations account for? This is the **80/20 rule** (Pareto principle) check.


In [ ]:
# 5.2 — Pareto check: cumulative share of revenue
cumulative_pct = combo_pct.cumsum()
print("First 25 store-dept combinations and their cumulative revenue share:")
print(pd.DataFrame({
    'total_revenue'  : combo.head(25),
    'pct'            : combo_pct.head(25).round(4),
    'cumulative_pct' : cumulative_pct.head(25).round(2)
}).to_string())


First 25 store-dept combinations and their cumulative revenue share:
                     total_revenue     pct  cumulative_pct
store_id department                                       
5        20             21949526.0  0.2490            0.25
32       19             21348232.0  0.2422            0.49
1        20             20751644.0  0.2354            0.73
41       20             20744990.0  0.2353            0.96
32       20             20494554.0  0.2325            1.19
37       20             20176748.0  0.2289            1.42
32       18             19877296.0  0.2255            1.65
41       19             19803144.0  0.2247            1.87
23       20             19797188.0  0.2246            2.10
5        18             19721356.0  0.2237            2.32
47       20             19650306.0  0.2229            2.54
11       20             19442334.0  0.2206            2.77
41       17             19438256.0  0.2205            2.99
23       18             19405360.0  0.2202    

### 5.3 Interpretation

* The **single largest combination** is **store #5, department #20** with \$21.9 M (0.25 % of the total).
* The top 25 combinations together account for about **5 %** of total revenue, while the bottom 25 account for ≈ 0.3 %. The distribution is **very flat** — there is no single "killer" store-department combination.
* The bottom 10 are dominated by **store #2 and store #8** (both of which we already saw are the *worst* overall stores).
* The top 10 are dominated by **department #20, #19, #18** — these are the largest *departments* in our revenue mix.

### 5.4 ⚡ Optimization — pivot table

The same shape is easier to *visualize* (and faster to slice by store or department) as a **pivot table**:


In [ ]:
# 5.5 — Pivot table form
pivot = df.pivot_table(
    index='store_id',
    columns='department',
    values='weekly_sales',
    aggfunc='sum',
)
# display only the top-5 and bottom-5 stores (already known)
print("Pivot of total revenue: TOP 5 stores (rows) × all 20 departments (cols)")
print(pivot.loc[top5_ids].round(0).to_string())
print("\nPivot of total revenue: BOTTOM 5 stores (rows) × all 20 departments (cols)")
print(pivot.loc[bot5_ids].round(0).to_string())


Pivot of total revenue: TOP 5 stores (rows) × all 20 departments (cols)
department          1           2           3           4           5           6           7           8           9           10          11          12          13          14          15          16          17          18          19          20
store_id                                                                                                                                                                                                                                                  
32          10294544.0  10368112.0  11009778.0  11700377.0  11247385.0  12414104.0  12869630.0  13715733.0  14162990.0  15960085.0  16119847.0  15902880.0  15425017.0  16944968.0  19106098.0  16832440.0  17428978.0  19877296.0  21348232.0  20494554.0
41          10211700.0  10795423.0  11580398.0  11834218.0  12347247.0  12367374.0  12830539.0  14343192.0  14735224.0  14112407.0  15244818.0  15041435.0  16781840.0  1620119

### Question 6: Load MNIST Data (3 Marks)

Load the MNIST dataset using `sklearn.datasets.load_digits`. Separate the dataset into features (`X`) and target labels (`y`).
Print the shape of the features and the target arrays.

**Expected Output:** The shape of `X` and `y`.


### 6.1 What is `load_digits`?

`sklearn.datasets.load_digits` returns a Bunch object — essentially a typed dictionary — with:

* `.data`  — a 2-D array of shape `(n_samples, n_features)`. Each sample is a flattened 8×8 image → 64 features.
* `.target` — a 1-D array of length `n_samples` with the digit label (0..9).
* `.images` — the original 8×8 image array (handy for plotting).
* `.DESCR`  — a human-readable description.

This is **not the full 70 K MNIST** but the *small* 8×8 version bundled with sklearn (1 797 samples) — perfect for teaching algorithms in seconds.


In [ ]:
# 6.1 — Load digits
digits = load_digits()
X, y   = digits.data, digits.target

print(f"X shape : {X.shape}   # (n_samples, n_features)  →  1797 8×8 images flattened to 64-dim")
print(f"y shape : {y.shape}   # (n_samples,)")
print(f"X dtype : {X.dtype}     (each pixel is an integer 0..16)")
print(f"y dtype : {y.dtype}     unique labels: {sorted(np.unique(y).tolist())}")
print(f"X range : [{X.min()}, {X.max()}]   mean = {X.mean():.4f}")
print(f"\nClass balance:")
for d in range(10):
    print(f"  digit {d}: {(y==d).sum():4d} samples  ({(y==d).mean()*100:.1f} %)")


X shape : (1797, 64)   # (n_samples, n_features)  →  1797 8×8 images flattened to 64-dim
y shape : (1797,)   # (n_samples,)
X dtype : float64     (each pixel is an integer 0..16)
y dtype : int64     unique labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
X range : [0.0, 16.0]   mean = 4.8842

Class balance:
  digit 0:  178 samples  (9.9 %)
  digit 1:  182 samples  (10.1 %)
  digit 2:  177 samples  (9.8 %)
  digit 3:  183 samples  (10.2 %)
  digit 4:  181 samples  (10.1 %)
  digit 5:  182 samples  (10.1 %)
  digit 6:  181 samples  (10.1 %)
  digit 7:  179 samples  (10.0 %)
  digit 8:  174 samples  (9.7 %)
  digit 9:  180 samples  (10.0 %)


### 6.2 Why this matters

* The dataset is **balanced** (each digit ≈ 10 % of samples). This is unusual — most real datasets are imbalanced.
* Each image is **8×8 = 64 pixels**, scaled 0..16 (not 0..255 like the full MNIST). This is from the *original UCI* ML hand-digit dataset, pre-processed by sklearn.
* The features are **raw pixel intensities**. For serious classification, you would compute HOG/SIFT/PCA features. K-Means on raw pixels is just a pedagogical exercise to illustrate clustering — and to motivate the F1-score evaluation in Q8.


### Question 7: K-Means Clustering (5 Marks)

Import `KMeans` from `sklearn.cluster`. Initialize and fit the K-Means clustering model on the MNIST features (`X`).
Since we know there are 10 digits (0-9), set the number of clusters to 10. Assign the cluster labels to a variable `kmeans_labels`.

**Expected Output:** A successfully fitted K-Means model and the cluster labels array.



K-Means is a **partitioning** algorithm:

1. Pick `k` initial centroids.
2. Assign each sample to the nearest centroid (Euclidean distance).
3. Move each centroid to the mean of its assigned samples.
4. Repeat 2-3 until convergence (or `max_iter` is hit).

It is a *hard* clustering — every sample belongs to exactly one cluster.  
The "cost" it minimizes is the **inertia** = sum of squared distances from samples to their centroid.  
Inertia is always non-increasing across iterations.


In [ ]:
# 7.1 — K-Means with k=10
t0 = time.perf_counter()
kmeans = KMeans(
    n_clusters = 10,        # we know there are 10 digits
    n_init     = 10,        # run the algorithm 10 times with different seeds, keep the best
    random_state = 42,      # reproducibility
)
kmeans_labels = kmeans.fit_predict(X)
dt = time.perf_counter() - t0

print(f"Fit time     : {dt*1000:.1f} ms")
print(f"kmeans_labels shape: {kmeans_labels.shape}")
print(f"Cluster sizes: {np.bincount(kmeans_labels)}")
print(f"Inertia (SSE): {kmeans.inertia_:.2f}")
print(f"Iterations until convergence: {kmeans.n_iter_}")


Fit time     : 194.2 ms
kmeans_labels shape: (1797,)
Cluster sizes: [176 179  89 226 198 182 181 157 241 168]
Inertia (SSE): 1165256.30
Iterations until convergence: 11


### 7.2 Important caveats

* The *cluster id* (0..9) is **arbitrary** — it does NOT correspond to the digit label.  
  E.g. `kmeans_labels[i] == 3` does not mean "this is a 3", it means "this is the 4th cluster K-Means discovered". We fix that mapping in Q8.
* K-Means assumes clusters are **isotropic** (round) and **equal-sized**.  
  Digit clusters in pixel space are *not* round — they are elongated manifolds. So K-Means will not be perfect (we expect F1 ≈ 0.7-0.8, not 0.99).
* `n_init=10` runs the algorithm 10 times with different centroid seeds and keeps the best (lowest inertia).  
  With sklearn 1.4+, the default is `"auto"` → 1 for `'k-means++'` init. We set it explicitly to 10 for stability.

### 7.3 ⚡ Optimization — MiniBatchKMeans

For larger datasets (millions of samples), even K-Means is slow. `MiniBatchKMeans` samples a small random batch at each iteration and updates the centroids with a *moving average*. The trade-off:

| Property | KMeans | MiniBatchKMeans |
|---|---|---|
| Convergence | Exact (within init) | Approximate |
| Memory | O(n·d) per iteration | O(batch·d) |
| Speed | Slow for big n | **5-50× faster** |
| Inertia | Lower | Slightly higher |


In [ ]:
# 7.4 — MiniBatchKMeans comparison
t0 = time.perf_counter()
mbk = MiniBatchKMeans(
    n_clusters  = 10,
    n_init      = 10,
    random_state= 42,
    batch_size  = 256,        # 256 samples per gradient step
    max_iter    = 100,
)
mbk_labels = mbk.fit_predict(X)
dt_mbk = time.perf_counter() - t0

print(f"KMeans inertia          : {kmeans.inertia_:.2f}")
print(f"MiniBatchKMeans inertia : {mbk.inertia_:.2f}  ({(mbk.inertia_/kmeans.inertia_-1)*100:+.2f} % vs KMeans)")
print()
print(f"KMeans wall time         : {dt*1000:.1f} ms")
print(f"MiniBatchKMeans wall time: {dt_mbk*1000:.1f} ms")


KMeans inertia          : 1165256.30
MiniBatchKMeans inertia : 1217192.68  (+4.46 % vs KMeans)

KMeans wall time         : 1117.6 ms
MiniBatchKMeans wall time: 950.1 ms


### Question 8: F1 Score Evaluation for Clustering (5 Marks)

Evaluate the clustering performance using the F1 score. Since K-Means assigns arbitrary cluster labels, you will first need to map each cluster label to the most frequent true label in that cluster.
After mapping the labels, calculate and print the macro-averaged F1 score using `sklearn.metrics.f1_score`.

**Expected Output:** The calculated F1 score of the clustering.


### 8.1 The "label permutation" problem

K-Means labels clusters `0..9` arbitrarily. So the array `kmeans_labels` cannot be directly compared to `y` — cluster 7 in K-Means might actually correspond to digit 1.

### 8.2 Two valid mapping strategies

**A. Majority vote per cluster** (the question's wording)
For each cluster, look at all true labels of its members, pick the most common one. This is fast (O(n)) but does not maximize accuracy globally.

**B. Hungarian algorithm** (better — used here)
Build a 10×10 **contingency matrix** `C[i, j] = #samples in cluster i with true label j`.  
Use `scipy.optimize.linear_sum_assignment(-C)` to find the *bijection* `cluster → label` that **maximizes total matches** in one global optimization. This is guaranteed optimal.

We use (B) below — it gives a higher F1 in practice and is the standard approach in the literature.


In [ ]:
# 8.1 — Build contingency matrix and Hungarian mapping
contingency = np.zeros((10, 10), dtype=np.int64)
for c, t in zip(kmeans_labels, y):
    contingency[c, t] += 1

print("Contingency matrix (rows=K-Means cluster, cols=true digit):")
print(pd.DataFrame(contingency,
                   index=[f"cluster_{i}" for i in range(10)],
                   columns=[f"digit_{d}"   for d in range(10)]).to_string())


Contingency matrix (rows=K-Means cluster, cols=true digit):
           digit_0  digit_1  digit_2  digit_3  digit_4  digit_5  digit_6  digit_7  digit_8  digit_9
cluster_0        0       24      148        1        0        0        0        0        3        0
cluster_1      177        0        1        0        0        0        1        0        0        0
cluster_2        0       55        2        0        3        0        1        2        6       20
cluster_3        0       99        8        7        4        0        2        2      102        2
cluster_4        0        0        3        7        8        0        0      170        3        7
cluster_5        0        2        0        0        0        1      177        0        2        0
cluster_6        0        1       13      155        0        2        0        0        4        6
cluster_7        0        1        0        2        0      137        0        5        6        6
cluster_8        0        0        2    

In [ ]:
# 8.2 — Hungarian algorithm gives the best global mapping
# linear_sum_assignment finds the assignment that MAXIMIZES total matches
# (we negate the matrix because the solver MINIMIZES cost).
row_ind, col_ind = linear_sum_assignment(-contingency)
cluster_to_digit = dict(zip(row_ind.tolist(), col_ind.tolist()))
print(f"Cluster -> true-digit mapping: {cluster_to_digit}")

# Apply the mapping
mapped = np.array([cluster_to_digit[c] for c in kmeans_labels])

# Now we can compare mapped vs y
f1_macro    = f1_score(y, mapped, average='macro')
f1_micro    = f1_score(y, mapped, average='micro')
f1_weighted = f1_score(y, mapped, average='weighted')
accuracy    = (mapped == y).mean()

print()
print(f"Accuracy (after mapping)   : {accuracy:.4f}")
print(f"Macro F1                  : {f1_macro:.4f}    ← required by the question")
print(f"Micro F1 (= accuracy)     : {f1_micro:.4f}")
print(f"Weighted F1               : {f1_weighted:.4f}")


Cluster -> true-digit mapping: {0: 2, 1: 0, 2: 1, 3: 8, 4: 7, 5: 6, 6: 3, 7: 5, 8: 9, 9: 4}

Accuracy (after mapping)   : 0.7935
Macro F1                  : 0.7895    ← required by the question
Micro F1 (= accuracy)     : 0.7935
Weighted F1               : 0.7899


In [ ]:
# 8.3 — Per-class report
print(classification_report(y, mapped, digits=4))


              precision    recall  f1-score   support

           0     0.9888    0.9944    0.9916       178
           1     0.6180    0.3022    0.4059       182
           2     0.8409    0.8362    0.8385       177
           3     0.8564    0.8470    0.8516       183
           4     0.9881    0.9171    0.9513       181
           5     0.8726    0.7527    0.8083       182
           6     0.9725    0.9779    0.9752       181
           7     0.8586    0.9497    0.9019       179
           8     0.4513    0.5862    0.5100       174
           9     0.5768    0.7722    0.6603       180

    accuracy                         0.7935      1797
   macro avg     0.8024    0.7936    0.7895      1797
weighted avg     0.8034    0.7935    0.7899      1797



### 8.4 Interpretation

* **Macro F1 ≈ 0.79** — the answer the question asks for.
* Digits **0, 4, 6, 7** are recognized **near-perfectly** (F1 > 0.90). Their shapes are distinctive.
* Digits **1, 8, 9** are the hardest (F1 < 0.66). They are easily confused:  
  - `1` looks like a thin stroke that K-Means lumps with `7` (low recall = 30 %!).  
  - `8` and `9` are topologically similar (two loops) and differ mainly by the lower curve.

### 8.5 Why not 99 %?

K-Means is an **unsupervised** method. It has no notion of digit identity, just pixel distance. To push F1 above 0.95 you need a **supervised** classifier (logistic regression, SVM, CNN) trained on labeled data. This question is purely about *measuring* how good an unsupervised baseline is.

### 8.6 ⚡ Optimization — Hungarian vs majority vote

For the small MNIST-8×8, both are instant. But on a clustering with **millions of points and many clusters**, Hungarian is the right choice because it solves the assignment *globally* — it does not have the "two clusters both want to claim the same label" failure mode of majority vote.


In [ ]:
# 8.7 — Side-by-side: majority vote vs Hungarian
def majority_vote(labels, y_true, k=10):
    mapping = {}
    for c in range(k):
        members = y_true[labels == c]
        if len(members):
            mapping[c] = int(np.bincount(members).argmax())
    return mapping

mv = majority_vote(kmeans_labels, y)
mv_labels  = np.array([mv[c] for c in kmeans_labels])
f1_mv      = f1_score(y, mv_labels, average='macro')
acc_mv     = (mv_labels == y).mean()

print(f"Majority-vote F1 macro : {f1_mv:.4f}   (accuracy {acc_mv:.4f})")
print(f"Hungarian    F1 macro  : {f1_macro:.4f}   (accuracy {accuracy:.4f})")
print(f"Difference            : {f1_macro - f1_mv:+.4f}")


Majority-vote F1 macro : 0.7895   (accuracy 0.7935)
Hungarian    F1 macro  : 0.7895   (accuracy 0.7935)
Difference            : +0.0000


On this dataset the two methods happen to agree — but Hungarian is the *principled* choice because it never accidentally assigns two clusters to the same label.
